In [1]:
import requests
import pandas as pd
from datetime import datetime
from google.colab import files

# 1. Cấu hình tham số
ticker = "FPT"
start_date = "2025-01-01"
end_date = "2026-06-08"

# Chuyển đổi định dạng ngày sang Timestamp
start_ts = int(datetime.strptime(start_date, "%Y-%m-%d").timestamp())
end_ts = int(datetime.strptime(end_date, "%Y-%m-%d").timestamp())

# 2. Đường link API lấy dữ liệu lịch sử của TCBS
url = f"https://apipubtracks.tcbs.com.vn/api/v1/derivative/candles?ticker={ticker}&type=stock&from={start_ts}&to={end_ts}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    # 3. Gửi yêu cầu lấy dữ liệu
    response = requests.get(url, headers=headers, timeout=15)

    if response.status_code == 200:
        data = response.json()

        if "data" in data and len(data["data"]) > 0:
            # 4. Chuyển đổi dữ liệu JSON thành DataFrame
            raw_df = pd.DataFrame(data["data"])

            df = pd.DataFrame()
            df['time'] = pd.to_datetime(raw_df['t'], unit='s').dt.strftime('%Y-%m-%d')
            df['open'] = raw_df['o']
            df['high'] = raw_df['h']
            df['low'] = raw_df['l']
            df['close'] = raw_df['c']
            df['volume'] = raw_df['v']

            df = df.sort_values('time').reset_index(drop=True)

            # 5. In thử kiểm tra kết quả
            print("--- Kết quả tải dữ liệu thành công ---")
            print(df.head())

            # 6. Xuất thành file CSV trên bộ nhớ Colab
            df.to_csv('FPT_stock_data.csv', index=False)
            print("\n--- Đã tạo xong file FPT_stock_data.csv ---")

            # 7. ÉP TRÌNH DUYỆT TỰ ĐỘNG TẢI FILE VỀ MÁY TÍNH
            print("Đang kích hoạt tải xuống máy tính...")
            files.download('FPT_stock_data.csv')

        else:
            print("Không tìm thấy dữ liệu trong khoảng thời gian này.")
    else:
        print(f"Lỗi kết nối đến máy chủ TCBS. Mã lỗi: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"Lỗi mạng hệ thống: {e}")

Lỗi mạng hệ thống: HTTPSConnectionPool(host='apipubtracks.tcbs.com.vn', port=443): Max retries exceeded with url: /api/v1/derivative/candles?ticker=FPT&type=stock&from=1735689600&to=1780876800 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7c0c579f34d0>: Failed to resolve 'apipubtracks.tcbs.com.vn' ([Errno -2] Name or service not known)"))
